# Preface - fractional TLs (meanTL) to Diet Composition

In 1986's article, the calculation was done in the global level - all fish were assumed to be on TL=2, with TE=0.1.  
In 1995's article the calculation was done with fractional TL for each group of Fish from similar TL. TE was still assumed to be 0.1 globally.  
In the notebook 1995_group2species.ipynb we explore the effect of this aggregation, and see that it doesn't have a significant effect.  

In this notebook we'd check the effect of using fractional TLs, compared to using diet composition to calcualte PPR.
1. we still use global TE=0.1
2. we assume each fish only eats from 2 TLs under it.

Since it seems like the "species group" used in 1995's article are not either commercial or functional groups,  
we will perform the analysis on a sample catch area (the Baltic Sea), and see how converting groups to species can increase PPR.

In [2]:
import pandas as pd

# Database

We are using the same database loaded in 1995_group2species.ipynb;  
It contains catches from the Baltic sea in the years 1988-1991, and TLs taken from 2020's supplementary material.

In [4]:
interesting_cols = [
    'scientific_name',
    'common_name',
    'habitat',
    'tonnes',
    'mean_TL',
]
df = pd.read_excel("Baltic_88-91_group2species.xlsx", usecols=interesting_cols)
df

,scientific_name,common_name,tonnes,habitat,mean_TL
0,Gadus morhua,Atlantic cod,9.858257e+05,benthopelagic,4.09
1,Clupea harengus,Atlantic herring,1.607137e+06,benthopelagic,3.38
2,Perca fluviatilis,European perch,2.821741e+04,demersal,4.35
3,Salmo salar,Atlantic salmon,1.963040e+04,benthopelagic,4.50
4,Sprattus sprattus,European sprat,3.711686e+05,pelagic-neritic,3.01
5,Esox lucius,Northern pike,1.767858e+04,demersal,4.07
6,Platichthys flesus,European flounder,3.103626e+04,demersal,3.32
7,Belone belone,Garfish,4.131295e+03,pelagic-oceanic,4.16
8,Osmerus eperlanus,European smelt,1.977640e+04,pelagic-neritic,3.46
9,Sander lucioperca,Pike-perch,5.035287e+03,pelagic,4.04


# Calculations

In [7]:
import numpy as np

# Define global constants:
TE = 0.1  # global TE, also possible to use 0.119 from 2020's article.

def SPPR(meanTL):
    TL_fraction = meanTL % 1
    TL_int = int(meanTL)
    sppr = (1-TL_fraction) * (1/TE)**(TL_int-1) + TL_fraction * (1/TE)**(TL_int)
    return sppr

In [8]:
df['SPPR'] = df['mean_TL'].apply(SPPR)
df['PPR'] = df['tonnes'] * df['SPPR']

df['classic SPPR'] = df['mean_TL'].apply(lambda x: (1/TE)**(x-1))
df['classic PPR'] = df['tonnes'] * df['classic SPPR']

df

,scientific_name,common_name,tonnes,habitat,mean_TL,SPPR,PPR,classic SPPR,classic PPR
0,Gadus morhua,Atlantic cod,9.858257e+05,benthopelagic,4.09,1810.0,1.784345e+09,1230.268771,1.212831e+09
1,Clupea harengus,Atlantic herring,1.607137e+06,benthopelagic,3.38,442.0,7.103545e+08,239.883292,3.855253e+08
2,Perca fluviatilis,European perch,2.821741e+04,demersal,4.35,4150.0,1.171022e+08,2238.721139,6.317090e+07
3,Salmo salar,Atlantic salmon,1.963040e+04,benthopelagic,4.50,5500.0,1.079672e+08,3162.277660,6.207679e+07
4,Sprattus sprattus,European sprat,3.711686e+05,pelagic-neritic,3.01,109.0,4.045738e+07,102.329299,3.798142e+07
5,Esox lucius,Northern pike,1.767858e+04,demersal,4.07,1630.0,2.881608e+07,1174.897555,2.077052e+07
6,Platichthys flesus,European flounder,3.103626e+04,demersal,3.32,388.0,1.204207e+07,208.929613,6.484393e+06
7,Belone belone,Garfish,4.131295e+03,pelagic-oceanic,4.16,2440.0,1.008036e+07,1445.439771,5.971538e+06
8,Osmerus eperlanus,European smelt,1.977640e+04,pelagic-neritic,3.46,514.0,1.016507e+07,288.403150,5.703575e+06
9,Sander lucioperca,Pike-perch,5.035287e+03,pelagic,4.04,1360.0,6.847990e+06,1096.478196,5.521083e+06


In [9]:
df.to_excel('Baltic_88-91_TL2structure.xlsx', index=False)